## 1. Setup

In [ ]:
import platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from joblib import Parallel, delayed
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# LASSO at tiny penalties on noisy returns
# warnings.filterwarnings("ignore", category=ConvergenceWarning)

CONFIG = {
    "validation_start": "2012-02-01",  # the first day with 250 rows of features before it (they start on 2011-02-01)
    "validation_end": "2017-12-31",
    "test_start": "2018-01-01",
    "test_end": "2025-12-31",
    "train_windows": [125, 250],  # training window L, in trading days
    "refit_intervals": [5, 10, 21],  # trading days between refits
    "embargo_rows": 2,  # row t-1's label needs Open[t+1], unknown at the close of t
    "cv_folds": 5,
    "lasso_n_penalties": 20,
    "lasso_min_penalty_ratio": 0.05,  # penalty grid: from the largest (keeps nothing) down to 5% of it
    "ridge_penalties": [float(penalty) for penalty in np.logspace(-1, 5, 13)],
    "ar_lags": 3,
    "feature_clip": 5.0,  # standardised features are clipped to +-5
    "label_winsor_quantiles": [0.01, 0.99],  # for train labels winsorization
    "min_train_row_share": 0.8,  # skip a refit if fewer than 80% of the window's rows are usable
    # "n_jobs": -1,
    "n_jobs": -2,
}

TEAM_REPO_DIR = Path("dsa5205_final_project_team_3")
PREPARED_DATA_DIR = TEAM_REPO_DIR / "data" / "prepared"

MODELS = ["lasso", "ridge", "ar", "mean"]
MODEL_NAMES = {
    "lasso": "LASSO",
    "ridge": "Ridge",
    "ar": "AR(3)",
    "mean": "Historical mean",
}

print("python", platform.python_version())
for module in (np, pd, sklearn):
    print(f"{module.__name__:12s} {module.__version__}")

## 2. Data

In [ ]:
MODEL_DATA_FILE = PREPARED_DATA_DIR / "model_data.csv"
CLEAN_PRICES_FILE = PREPARED_DATA_DIR / "clean_prices.csv"
FEATURE_COLUMNS = [
    "return_1d",
    "return_5d",
    "return_20d",
    "ma_gap_20d",
    "volatility_20d",
    "intraday_range",
    "volume_ratio_20d",
]

model_data = pd.read_csv(
    MODEL_DATA_FILE, parse_dates=["Date", "entry_date", "exit_date"]
)
TICKERS = sorted(model_data["Ticker"].unique())
N_STOCKS = len(TICKERS)
CALENDAR = pd.DatetimeIndex(
    pd.read_csv(CLEAN_PRICES_FILE, usecols=["Date"], parse_dates=["Date"])[
        "Date"
    ].unique()
).sort_values() 


# One wide frame (signal dates x stocks) per column of the team's long table.
def to_wide_frame(column):
    return model_data.pivot(index="Date", columns="Ticker", values=column)[
        TICKERS
    ].sort_index()


SIGNAL_DATES = to_wide_frame("target_return").index
N_DAYS = len(SIGNAL_DATES)
rows_by_date = model_data.groupby("Date")
next_trading_day = pd.Series(CALENDAR[1:], index=CALENDAR[:-1])
ENTRY_DATES = pd.DatetimeIndex(rows_by_date["entry_date"].first().reindex(SIGNAL_DATES))
EXIT_DATES = pd.DatetimeIndex(rows_by_date["exit_date"].first().reindex(SIGNAL_DATES))

## 3. Features and labels

In [ ]:
OWN_FEATURES = [
    feature for feature in FEATURE_COLUMNS if feature != "return_1d"
] 

FEATURES = {feature: to_wide_frame(feature) for feature in FEATURE_COLUMNS}
# Labels: the open-to-open return from t+1 to t+2 (dividend-adjusted).
LABELS = to_wide_frame("target_return")
# AR(3) inputs: the stock's own close-to-close returns of days t, t-1, t-2.
AR_INPUTS = {
    f"return_lag{lag}": FEATURES["return_1d"].shift(lag)
    for lag in range(CONFIG["ar_lags"])
}


def to_float_array(frame):
    values = frame.to_numpy(dtype=float, copy=True)
    values[~np.isfinite(values)] = np.nan
    return values


CROSS_RETURNS = to_float_array(FEATURES["return_1d"])  # days x stocks
OWN_INDICATORS = np.stack(
    [to_float_array(FEATURES[feature]) for feature in OWN_FEATURES], axis=2
)  # days x stocks x 6
AR_RETURNS = np.stack(
    [to_float_array(AR_INPUTS[lag_name]) for lag_name in sorted(AR_INPUTS)], axis=2
)  # days x stocks x 3
LABEL_ARRAY = to_float_array(LABELS)  # days x stocks

FIRST_REFIT_ROW = int(
    SIGNAL_DATES.searchsorted(pd.Timestamp(CONFIG["validation_start"]))
)
FORECAST_END_ROW = int(
    SIGNAL_DATES.searchsorted(pd.Timestamp(CONFIG["test_end"]), side="right")
)
# Refit rows for each refit interval
REFIT_ROWS = {
    refit_interval: np.arange(FIRST_REFIT_ROW, FORECAST_END_ROW, refit_interval)
    for refit_interval in CONFIG["refit_intervals"]
}

## 4. Rolling estimation

In [ ]:
def standardize(train_features, new_features, clip_at):
    # StandardScaler ignores missing values and constant columns get scale 1.
    scaler = StandardScaler().fit(train_features)
    scale_and_clip = lambda features: np.nan_to_num(
        np.clip(scaler.transform(features), -clip_at, clip_at), nan=0.0
    )
    return scale_and_clip(train_features), scale_and_clip(new_features)


def run_stock(
    model,
    window,
    config,
    cross_returns,
    own_indicators,
    ar_returns,
    labels,
    refit_rows,
    end_row,
):
    """Rolling fits for one stock; returns its daily forecasts (NaN where no model was fitted).

    model          : "lasso", "ridge", "ar" or "mean"
    window         : training window L, in rows (trading days)
    cross_returns  : days x stocks, every universe stock's return on day t (the same for every target stock)
    own_indicators : days x 6 technical indicators of this stock
    ar_returns     : days x 3 returns of this stock on days t, t-1 and t-2
    labels         : this stock's labels, the open-to-open return from t+1 to t+2
    refit_rows     : rows at which the model is refitted
    end_row        : forecasts stop before this row
    """
    # Features for each model. LASSO and ridge see every stock's return plus this stock's indicators;
    # AR(3) sees only this stock's last three returns; the historical mean needs none (None).
    features = None
    if model == "lasso":
        features = np.hstack([cross_returns, own_indicators])
    elif model == "ridge":
        features = np.hstack([cross_returns, own_indicators])
    elif model == "ar":
        features = ar_returns
    elif model == "mean":
        pass

    # Forward folds for choosing the penalty, with a gap as long as the embargo.
    cv_folds = TimeSeriesSplit(n_splits=config["cv_folds"], gap=config["embargo_rows"])
    forecasts = np.full(len(labels), np.nan)

    for refit_index, refit_row in enumerate(refit_rows):
        # --- 1. Training rows: the L rows that end just before the refit (the embargo) ---
        last_train_row = refit_row - config["embargo_rows"]
        first_train_row = last_train_row - window + 1
        if first_train_row < 0:
            # not enough history yet
            continue
        train_rows = np.arange(first_train_row, last_train_row + 1)

        # Drop rows whose label or own indicators are missing
        train_rows = train_rows[
            np.isfinite(labels[train_rows])
            & np.isfinite(own_indicators[train_rows]).all(axis=1)
        ]
        # Skip this refit if too few are left
        if len(train_rows) < config["min_train_row_share"] * window:
            continue
        # no label unknown at the close of the refit day
        assert train_rows.max() <= refit_row - config["embargo_rows"]

        # --- 2. Forecast rows: from this refit up to the next one; the coefficients stay frozen ---
        next_refit_row = (
            refit_rows[refit_index + 1]
            if refit_index + 1 < len(refit_rows)
            else end_row
        )
        forecast_rows = np.arange(refit_row, next_refit_row)

        # --- 3. Training labels, winsorised at the window's 1st and 99th percentiles ---
        train_labels = np.clip(
            labels[train_rows],
            *np.quantile(labels[train_rows], config["label_winsor_quantiles"])
        )

        # Historical mean: the forecast is simply the average training label.
        if model == "mean":
            forecasts[forecast_rows] = train_labels.mean()
            continue

        # --- 4. Scale with training-window statistics, then fit and forecast ---
        train_scaled, forecast_scaled = standardize(
            features[train_rows], features[forecast_rows], config["feature_clip"]
        )

        if model == "lasso":
            # Penalty chosen by forward-fold CV over 20 values, from the largest (keeps nothing)
            # down to 5% of it.
            fitted = LassoCV(
                alphas=config["lasso_n_penalties"],
                eps=config["lasso_min_penalty_ratio"],
                cv=cv_folds,
                precompute=True,
                max_iter=5000,
            ).fit(train_scaled, train_labels)
        elif model == "ridge":
            # Same folds, scored by mean squared error, as LassoCV is.
            fitted = RidgeCV(
                alphas=config["ridge_penalties"],
                cv=cv_folds,
                scoring="neg_mean_squared_error",
            ).fit(train_scaled, train_labels)
        else:
            # AR(3): ordinary least squares on the stock's own last three returns.
            fitted = LinearRegression().fit(train_scaled, train_labels)
        forecasts[forecast_rows] = fitted.predict(forecast_scaled)

    return forecasts


# One forecast frame (signal dates x stocks) per model, window and refit interval.
# The stocks are fitted in parallel.
fit_settings = [
    (model, window, refit_interval)
    for model in MODELS
    for window in CONFIG["train_windows"]
    for refit_interval in CONFIG["refit_intervals"]
]
FORECASTS = {}
for model, window, refit_interval in tqdm(fit_settings, desc="fits"):
    per_stock_forecasts = Parallel(n_jobs=CONFIG["n_jobs"])(
        delayed(run_stock)(
            model,
            window,
            CONFIG,
            CROSS_RETURNS,
            OWN_INDICATORS[:, stock_index, :],
            AR_RETURNS[:, stock_index, :],
            LABEL_ARRAY[:, stock_index],
            REFIT_ROWS[refit_interval],
            FORECAST_END_ROW,
        )
        for stock_index in range(N_STOCKS)
    )
    FORECASTS[(model, window, refit_interval)] = pd.DataFrame(
        np.column_stack(per_stock_forecasts), index=SIGNAL_DATES, columns=TICKERS
    )

## 5. Forecast accuracy

In [ ]:
def r2(forecasts, dates, benchmark=None):
    realised = LABELS.loc[dates].to_numpy()
    predicted = forecasts.loc[dates].to_numpy()

    usable = np.isfinite(realised) & np.isfinite(predicted)
    if benchmark is not None:
        benchmark_values = benchmark.loc[dates].to_numpy()
        usable &= np.isfinite(benchmark_values)

    if benchmark is None:
        denominator = (realised[usable] ** 2).sum()
    else:
        denominator = ((realised - benchmark_values)[usable] ** 2).sum()

    return 1 - ((realised - predicted)[usable] ** 2).sum() / denominator

In [ ]:
VALIDATION_DATES = SIGNAL_DATES[
    (SIGNAL_DATES >= pd.Timestamp(CONFIG["validation_start"]))
    & (SIGNAL_DATES <= pd.Timestamp(CONFIG["validation_end"]))
]
print(
    f"Validation: {VALIDATION_DATES[0].date()} to {VALIDATION_DATES[-1].date()} ({len(VALIDATION_DATES)} days)"
)

table_rows = []
for (model, window, refit_interval), forecasts in FORECASTS.items():
    mean_forecasts = FORECASTS[("mean", window, refit_interval)]
    table_rows.append(
        {
            "model": MODEL_NAMES[model],
            "L": window,
            "R (days)": refit_interval,
            "R2 vs zero (%)": 100 * r2(forecasts, VALIDATION_DATES),
            "R2 vs hist. mean (%)": (
                100 * r2(forecasts, VALIDATION_DATES, mean_forecasts)
            ),
        }
    )
pd.DataFrame(table_rows).set_index(["model", "L", "R (days)"]).round(4)

In [ ]:
# Best window length and refit interval for LASSO:
chosen_window = 250
chosen_refit_interval = 5

In [ ]:
TEST_DATES = SIGNAL_DATES[
    (SIGNAL_DATES >= pd.Timestamp(CONFIG["test_start"]))
    & (SIGNAL_DATES <= pd.Timestamp(CONFIG["test_end"]))
]
print(
    f"Test: {TEST_DATES[0].date()} to {TEST_DATES[-1].date()} ({len(TEST_DATES)} days)"
)

table_rows = []
for (model, window, refit_interval), forecasts in FORECASTS.items():
    if window != chosen_window or refit_interval != chosen_refit_interval:
        continue

    mean_forecasts = FORECASTS[("mean", window, refit_interval)]
    table_rows.append(
        {
            "model": MODEL_NAMES[model],
            "L": window,
            "R (days)": refit_interval,
            "R2 vs zero (%)": 100 * r2(forecasts, TEST_DATES),
            "R2 vs hist. mean (%)": (100 * r2(forecasts, TEST_DATES, mean_forecasts)),
        }
    )
pd.DataFrame(table_rows).set_index(["model", "L", "R (days)"]).round(4)

In [ ]:
FORECASTS[("lasso", chosen_window, chosen_refit_interval)]